# 06 — Leakage-safe YOLO11 pseudo-label fine-tuning

This notebook extends the manually annotated segmentation set with
high-confidence masks predicted on additional VisA images.

## Pipeline position

`Teacher YOLO11 → filtered pseudo masks → combined training set → fine-tuned YOLO11`

## Leakage safeguards

- Pseudo-label confidence must be at least 0.90.
- The predicted product class must match the source class.
- Sources present in the supervised train, validation, or test export are
  excluded from pseudo-label generation.
- The original validation and test partitions remain untouched.
- Teacher and fine-tuned models are compared on the same test data.

The completed run generated 29,849 pseudo masks from 10,182 additional training
images.


## 1. Locate the scripted workflow

The underlying script handles pseudo-label generation, dataset assembly,
fine-tuning, Ultralytics validation, pixel-level mask evaluation, and metric
comparison.


In [ ]:
# Resolve the pseudo-label workflow and its artifact directory
from pathlib import Path
import subprocess
import sys

# Support launching from the historical experiment directory or its parent.
RIDAC_DIR = Path.cwd().resolve()
if not (RIDAC_DIR / 'pseudo_label_segmentation_retrain.py').is_file():
    candidate = RIDAC_DIR / 'misc' / 'ridac'
    if (candidate / 'pseudo_label_segmentation_retrain.py').is_file():
        RIDAC_DIR = candidate
    else:
        raise FileNotFoundError('Could not locate misc/ridac/pseudo_label_segmentation_retrain.py')
SCRIPT = RIDAC_DIR / 'pseudo_label_segmentation_retrain.py'
OUTPUT_ROOT = RIDAC_DIR / 'pseudo_label_segmentation_retraining'
print('Python:', sys.executable)
print('Workflow:', SCRIPT)


## 2. Reuse the completed run or execute expensive stages

The repository documents a completed experiment, so reuse is enabled by
default. With `REUSE_COMPLETED_RUN=True`, the workflow skips pseudo-label
generation and training, then refreshes evaluation outputs.

Set it to `False` only when deliberately rebuilding the experiment. That path
requires the source dataset, Roboflow export, prior full segmentation results,
teacher checkpoint, and a suitable GPU environment.


In [ ]:
# Build the workflow command and skip completed expensive stages by default
# False regenerates pseudo-labels and retrains from scratch.
REUSE_COMPLETED_RUN = True
command = [sys.executable, str(SCRIPT)]
# Reevaluate saved checkpoints without regenerating labels or training.
if REUSE_COMPLETED_RUN and (OUTPUT_ROOT / "run_summary.json").is_file():
    command += ["--skip-pseudo", "--skip-train"]
completed = subprocess.run(command, cwd=RIDAC_DIR, check=False)
if completed.returncode:
    raise RuntimeError(f"Retraining workflow exited with code {completed.returncode}")


## 3. Inspect run provenance and segmentation metrics

The summary records the teacher and trained checkpoint paths, pseudo-label
threshold, source-image counts, pseudo-mask count, validation/test sizes, and
seed.

The reports include:

- Box precision, recall, and mAP.
- Mask precision, recall, and mAP.
- Micro and macro pixel IoU/Dice.
- Boundary F1.
- Absolute teacher-to-fine-tuned changes.
- Per-class mask metrics.

Small negative changes are intentionally retained; model improvements should
not be reported selectively.


In [ ]:
# Load run provenance, overall changes, and per-class segmentation metrics
import json
import pandas as pd
from IPython.display import display

# Read saved reports only; this cell performs no model training.
overall = pd.read_csv(OUTPUT_ROOT / 'overall_segmentation_metrics.csv')
improvements = pd.read_csv(OUTPUT_ROOT / 'metric_improvements.csv')
per_class = pd.read_csv(OUTPUT_ROOT / 'per_class_mask_metrics.csv')
summary = json.loads((OUTPUT_ROOT / 'run_summary.json').read_text())
print(json.dumps(summary, indent=2))
display(overall.round(5))
display(improvements.round(5))
display(per_class.round(5))


## 4. Interpretation and production handoff

The completed run improved box and mask precision from approximately 0.9703 to
0.9903 and slightly improved box mAP@50:95. Mask mAP@50:95 decreased slightly,
while macro pixel IoU, Dice, and boundary F1 improved marginally.

The selected fine-tuned checkpoint is packaged at
`../models/yolo11_seg_best.pt` and is loaded by the Gradio application.
